In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import polars as pl
import pyspark
import psutil
import os
import tracemalloc
import time

In [2]:
#Ruta CSV
PATH_CSV = "/home/jassiel/Documentos/Labs/synthea/lab02"
archivos = ["patients.csv", "encounters.csv", "observations.csv"]

for archivo in archivos:
    filepath = os.path.join(PATH_CSV, archivo)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{archivo}: {size_mb:.2f} MB en disco")
    else:
        print(f"{archivo} no se encuentra en la ruta '{PATH_CSV}'")

patients.csv: 0.16 MB en disco
encounters.csv: 9.53 MB en disco
observations.csv: 68.77 MB en disco


Clasificación de columnas para patients.csv:
Identificadores: Id, SSN, DRIVERS, PASSPORT, FIRST, MIDDLE, LAST, MAIDEN, ADDRESS.   
Fechas: BIRTHDATE, DEATHDATE.   
Categóricas: PREFIX, SUFFIX, MARITAL, RACE, ETHNICITY, GENDER, BIRTHPLACE, CITY, STATE, COUNTY, FIPS, ZIP.   
Numéricas: LAT, LON, HEALTHCARE_EXPENSES, HEALTHCARE_COVERAGE, INCOME.   

In [3]:
#Actividad 1. Carga
def obtener_memoria_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 * 1024)

#Consumo pandas
memoria_antes = {}

for archivo in archivos:
    filepath = os.path.join(PATH_CSV, archivo)
    df_temp = pd.read_csv(filepath, on_bad_lines='skip')
    memoria_antes[archivo] = obtener_memoria_mb(df_temp)
    del df_temp

#dtypes explicitos
dtypes_patients = {
    'Id': 'string',
    'SSN': 'string',
    'DRIVERS': 'string',
    'PASSPORT': 'string',
    'PREFIX': 'category',
    'FIRST': 'string',
    'MIDDLE': 'string',
    'LAST': 'string',
    'SUFFIX': 'category',
    'MAIDEN': 'string',
    'MARITAL': 'category',
    'RACE': 'category',
    'ETHNICITY': 'category',
    'GENDER': 'category',
    'BIRTHPLACE': 'category',
    'ADDRESS': 'string',
    'CITY': 'category',
    'STATE': 'category',
    'COUNTY': 'category',
    'FIPS': 'category',
    'ZIP': 'category',
    'LAT': 'float32',
    'LON': 'float32',
    'HEALTHCARE_EXPENSES': 'float32',
    'HEALTHCARE_COVERAGE': 'float32',
    'INCOME': 'float32'
}
dates_patients = ['BIRTHDATE', 'DEATHDATE']

dtypes_encounters = {
    'Id': 'string',
    'PATIENT': 'category',
    'ORGANIZATION': 'category',
    'PROVIDER': 'category',
    'PAYER': 'category',
    'ENCOUNTERCLASS': 'category',
    'CODE': 'category',
    'DESCRIPTION': 'category',
    'BASE_ENCOUNTER_COST': 'float32',
    'TOTAL_CLAIM_COST': 'float32',
    'PAYER_COVERAGE': 'float32',
    'REASONCODE': 'category',
    'REASONDESCRIPTION': 'category'
}
dates_encounters = ['START', 'STOP']

dtypes_observations = {
    'PATIENT': 'category',
    'ENCOUNTER': 'category',
    'CATEGORY': 'category',
    'CODE': 'category',
    'DESCRIPTION': 'category',
    'VALUE': 'string',
    'UNITS': 'category',
    'TYPE': 'category'
}
dates_observations = ['DATE']

configuraciones = {
    "patients.csv": {"dtype": dtypes_patients, "parse_dates": dates_patients},
    "encounters.csv": {"dtype": dtypes_encounters, "parse_dates": dates_encounters},
    "observations.csv": {"dtype": dtypes_observations, "parse_dates": dates_observations}
}

memoria_despues = {}

for archivo in archivos:
    filepath = os.path.join(PATH_CSV, archivo)
    config = configuraciones[archivo]
    
    #Cargar directamente aplicando dtypes
    df_opt = pd.read_csv(
        filepath,
        dtype=config["dtype"],
        parse_dates=config["parse_dates"],
        on_bad_lines='skip'
    )
    memoria_despues[archivo] = obtener_memoria_mb(df_opt)
    del df_opt

#Tabla
tabla_entrega = pd.DataFrame({
    "Archivo": archivos,
    "Consumo Antes (MB)": [round(memoria_antes[a], 2) for a in archivos],
    "Consumo Después (MB)": [round(memoria_despues[a], 2) for a in archivos],
})

tabla_entrega["Ahorro (%)"] = round(
    ((tabla_entrega["Consumo Antes (MB)"] - tabla_entrega["Consumo Después (MB)"]) / tabla_entrega["Consumo Antes (MB)"]) * 100,
    2
)

print("Tabla de consumo antes y despues:")
print(tabla_entrega.to_string(index=False))

Tabla de consumo antes y despues:
         Archivo  Consumo Antes (MB)  Consumo Después (MB)  Ahorro (%)
    patients.csv                0.23                  0.16       30.43
  encounters.csv               11.66                  2.61       77.62
observations.csv               93.73                 23.09       75.37


In [4]:
#Actividad 2: Reducir memoria
#Carga gngenua de observation.csv
df_ingenua = pd.read_csv(filepath)

#Carga optimizada
observations = {
    'PATIENT': 'category',
    'ENCOUNTER': 'category',
    'CATEGORY': 'category',
    'CODE': 'category',
    'DESCRIPTION': 'category',
    'VALUE': 'string',
    'UNITS': 'category',
    'TYPE': 'category'
}

dtypes_observations = {
    'PATIENT': 'category',
    'ENCOUNTER': 'category',
    'CATEGORY': 'category',
    'CODE': 'category',
    'DESCRIPTION': 'category',
    'VALUE': 'string',
    'UNITS': 'category',
    'TYPE': 'category'
}
dates_observations = ['DATE']

df_optimizada = pd.read_csv(
    filepath,
    dtype=dtypes_observations,
    parse_dates=dates_observations,
    on_bad_lines='skip'
)

#Medición de memoria por columna
desglose = []
for col in df_ingenua.columns:
    mb_antes = df_ingenua[col].memory_usage(deep=True) / (1024 * 1024)
    mb_despues = df_optimizada[col].memory_usage(deep=True) / (1024 * 1024)
    
    desglose.append({
        "Columna": col,
        "dtype antes": str(df_ingenua[col].dtype),
        "dtype después": str(df_optimizada[col].dtype),
        "MB antes": round(mb_antes, 2),
        "MB después": round(mb_despues, 2),
        "Ahorro (%)": round(((mb_antes - mb_despues) / mb_antes) * 100, 2)
    })

df_desglose = pd.DataFrame(desglose)

print("Consumo por columna de observations.csv")
print(df_desglose.to_string(index=False))

mem_total_antes = df_ingenua.memory_usage(deep=True).sum() / (1024 * 1024)
mem_total_despues = df_optimizada.memory_usage(deep=True).sum() / (1024 * 1024)
reduccion_total = ((mem_total_antes - mem_total_despues) / mem_total_antes) * 100

print(f"\nReducción total alcanzada: {reduccion_total:.2f}%")

Consumo por columna de observations.csv
    Columna dtype antes dtype después  MB antes  MB después  Ahorro (%)
       DATE         str           str     10.94       10.94        0.00
    PATIENT         str      category     17.20        0.82       95.23
  ENCOUNTER         str      category     16.75        1.92       88.53
   CATEGORY         str      category      6.53        0.39       94.01
       CODE         str      category      5.69        0.79       86.07
DESCRIPTION         str      category     19.80        0.80       95.94
      VALUE         str        string      6.68        6.63        0.70
      UNITS         str      category      4.69        0.39       91.61
       TYPE         str      category      5.45        0.39       92.83

Reducción total alcanzada: 75.37%


¿Qué se pierde?
Date: Se asume un formato ISO/fecha estándar estricto, por lo que si vienen cadenas corruptas o con formatos heterogéneos, fallará o se convertirán en NaT.
Patient: Se pierde rendimiento al unir con valores nuevos, por lo que se generará sobrecarga de recategorización.
Encounter: Como los IDs de encuentros tienen valores únicos, el diccionario de categorías crece. Si casi no se repiten, category puede ser menos eficiente que string.
Code: Se pierde la capacidad de operar numéricamente sin convertir primero.
Description: Se pierden operaciones complejas de texto.
Value: Al mantenerlo como string se pierde la capacidad de hacer cálculos estadísticos directos sobre las mediciones cuantitativas.
Units: Al normalizar a un conjunto estricto de categorías se pueden crean categorías duplicadas.
Type: Se hace más rigido diccionario.

In [5]:
#Actividad 3: auditar la calidad de los datos
dtypes_patients = {'Id': 'string', 'GENDER': 'category', 'RACE': 'category', 'ETHNICITY': 'category'}
dates_patients = ['BIRTHDATE', 'DEATHDATE']

dtypes_encounters = {'Id': 'string', 'PATIENT': 'category', 'ENCOUNTERCLASS': 'category'}
dates_encounters = ['START', 'STOP']

dtypes_observations = {'PATIENT': 'category', 'ENCOUNTER': 'category', 'VALUE': 'string'}
dates_observations = ['DATE']

df_patients = pd.read_csv(os.path.join(PATH_CSV, "patients.csv"), dtype=dtypes_patients, parse_dates=dates_patients, on_bad_lines='skip')
df_encounters = pd.read_csv(os.path.join(PATH_CSV, "encounters.csv"), dtype=dtypes_encounters, parse_dates=dates_encounters, on_bad_lines='skip')
df_observations = pd.read_csv(os.path.join(PATH_CSV, "observations.csv"), dtype=dtypes_observations, parse_dates=dates_observations, on_bad_lines='skip')

#Elimiar la zona horaria para permitir comparaciones
df_encounters['START'] = pd.to_datetime(df_encounters['START'], utc=True).dt.tz_localize(None)
df_encounters['STOP'] = pd.to_datetime(df_encounters['STOP'], utc=True).dt.tz_localize(None)
df_patients['BIRTHDATE'] = pd.to_datetime(df_patients['BIRTHDATE'], utc=True).dt.tz_localize(None)
df_patients['DEATHDATE'] = pd.to_datetime(df_patients['DEATHDATE'], utc=True).dt.tz_localize(None)

#Porcentaje de valores faltantes por columna
print("valores faltantes por columna")
for nombre, df in zip(archivos, [df_patients, df_encounters, df_observations]):
    print(f"\n{nombre}")
    nulos = df.isnull().mean() * 100
    nulos_df = nulos[nulos > 0].reset_index()
    nulos_df.columns = ['Columna', '% Faltantes']
    if len(nulos_df) == 0:
        print("Sin valores faltantes.")
    else:
        print(nulos_df.to_string(index=False))

#Identificadores duplicados en patients.csv
duplicado_ids = df_patients['Id'].duplicated().sum()
print(f"\nDuplicados en patients.csv")
print(f"IDs de paciente duplicados: {duplicado_ids}")

#Coherencia temporal (Encuentros vs Nacimiento/Defunción)
df_enc_patients = df_encounters.merge(
    df_patients[['Id', 'BIRTHDATE', 'DEATHDATE']], 
    left_on='PATIENT', 
    right_on='Id', 
    how='left'
)

#Encuentros previos a fecha de nacimiento
enc_antes_nacimiento = df_enc_patients[df_enc_patients['START'] < df_enc_patients['BIRTHDATE']]

#Encuentros posteriores a fecha de defunción
enc_post_muerte = df_enc_patients[df_enc_patients['DEATHDATE'].notnull() & (df_enc_patients['START'] > df_enc_patients['DEATHDATE'])]

print(f"\nCoherencia temporal ")
print(f"Encuentros antes de nacer: {len(enc_antes_nacimiento)}")
print(f"Encuentros después de fallecer: {len(enc_post_muerte)}")

#Análisis de value en observations.csv
values_numerico = pd.to_numeric(df_observations['VALUE'], errors='coerce')
es_nulo_original = df_observations['VALUE'].isnull()
no_numericos = values_numerico.isnull() & (~es_nulo_original)

pct_no_num = (no_numericos.sum() / len(df_observations)) * 100

print(f"\nAnálisis de value en observations.csv")
print(f"Total observaciones: {len(df_observations)}")
print(f"Proporción no convertible a número: {pct_no_num:.2f}% ({no_numericos.sum()} filas)")

print("\nEjemplo de valores no numéricos encontrados en value:")
ejemplos_no_num = df_observations[no_numericos][['DESCRIPTION', 'VALUE']].drop_duplicates().head(10)
print(ejemplos_no_num.to_string(index=False))

valores faltantes por columna

patients.csv
            Columna  % Faltantes
          DEATHDATE    86.714542
            DRIVERS    18.132855
           PASSPORT    21.723519
             PREFIX    20.107720
             MIDDLE    18.671454
             SUFFIX    99.461400
             MAIDEN    72.710952
            MARITAL    31.956912
               FIPS    27.289048
                ZIP     0.179533
                LAT     0.179533
                LON     0.179533
HEALTHCARE_EXPENSES     0.179533
HEALTHCARE_COVERAGE     0.179533
             INCOME     0.179533

encounters.csv
            Columna  % Faltantes
              START     0.003329
               STOP     0.003329
            PATIENT     0.003329
       ORGANIZATION     0.003329
           PROVIDER     0.003329
              PAYER     0.003329
     ENCOUNTERCLASS     0.003329
               CODE     0.003329
        DESCRIPTION     0.003329
BASE_ENCOUNTER_COST     0.003329
   TOTAL_CLAIM_COST     0.003329
     PAYER_COVER

Valores faltantes:
- patients.csv: Tiene 86.71% de nulos en deathdate, lo cual es esperado, mientras que suffix y maiden presentan un 99.46% y 72.71% de datos nulos, respectivamente, lo cual tambieén es eperado.
- encounters.csv: reasoncode y reasondescription tienen 39.86% de valores nulos. Este valor puede entrar dentro de los valores normales de revisiones preventivas, chequeos o seguimientos donde no hay una patología o síntoma agudo.
- observatios.csv: Units tiene 27.78%, que corresponde a pruebas de laboratorio o varables cualitativas donde no hay un valor medible que entre en un rago.

Valores duplicados:
- patients.csv: No hay ids duplicados.

Coherencia temporal:
- Nacimiento: Hay 0 encuentros registrados antes del nacimiento.
- Defunción: Hay 72 encuentros registrados posteriores a la fecha de defunción del paciente.

Value en observations:
- Hay un 37.46% de datos que no se pueden convertir a número, en esta filas hay culificadores, diagnosticos de prescencia (positive o negativo para determinado laboratorio),  evaluación de risesgo y conducta (como hábitos o escalas clínicas).
- Este porcentaje es el justificante del por qué no se debe de pasar la coluimna value a float o int, ya que se eliminaria poco más de un tercio de la información de nuestro dataset.

In [6]:
#Actividad 4: unir tablas vlidando cardinalidades
#Tamaños originales
print(f"Filas originales en observations: {len(df_observations)}")
print(f"Filas originales en encounters: {len(df_encounters)}")

#Primer Merge: Observations (Left) -> Encounters (Right)
df_obs_enc = df_observations.merge(
    df_encounters,
    left_on='ENCOUNTER',
    right_on='Id',
    how='left',
    validate='m:1',
    indicator=True,
    suffixes=('', '_enc') #Para evitar colisión si hay columnas con el mismo nombre.
)

print(f"Filas después del primer merge: {len(df_obs_enc)}")
print("\nAuditoría del cruce (indicator):")
print(df_obs_enc['_merge'].value_counts())

#Limpiar la columna del indicador para el siguiente merge
df_obs_enc = df_obs_enc.drop(columns=['_merge'])

Filas originales en observations: 409805
Filas originales en encounters: 30043
Filas después del primer merge: 409805

Auditoría del cruce (indicator):
_merge
both          395236
left_only      14569
right_only         0
Name: count, dtype: int64


Primer merge: observatios + encounters.
- Laves de cruce: encounter (de observations) contra id (de encounters).
- Exlplicación clínica: Un encuentro genera multiples observaciones, una visita médica = toma de presión, peso, temperatura, etc,
- Cardinalidad: Muchos a uno = m:1
- Hay 14,569 observaciones que no encontraron correspondencia en la tabla de encuentros. Esto se explica con lo que se observó en la actividad 3, donde la columna encounters en observations.csv tiene 3.55% valores nulos.

In [7]:
print(f"Filas originales en patients: {len(df_patients)}")

#Segundo Merge: df_obs_enc (Left) -> Patients (Right)
df_final = df_obs_enc.merge(
    df_patients,
    left_on='PATIENT',
    right_on='Id',
    how='left',
    validate='m:1',
    indicator=True,
    suffixes=('', '_pat')
)

print(f"\nFilas después del segundo merge: {len(df_final)}")
print("\nAuditoría del cruce final (indicator):")
print(df_final['_merge'].value_counts())

mem_total_mb = df_final.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"\nMemoria total del dataset unido: {mem_total_mb:.2f} MB")

Filas originales en patients: 557

Filas después del segundo merge: 409805

Auditoría del cruce final (indicator):
_merge
both          409796
left_only          9
right_only         0
Name: count, dtype: int64

Memoria total del dataset unido: 357.75 MB


Segundo merge: obs_enc (del primer merge) + id (patients).
- Llaves de cruce: patient (primer merge) contra id (patient).
- Explicación clínica: Un paciente tiene multiples observaciones a lo largo de su historial clínico.
- Cardinalidad: muchos a uno = m:1.
- Hay 9 observaciones que no hicieron match, que representarias datos huérfanos, es es decir que no correspeonden a ningun id de paciente.

In [8]:
#Actividad 5: preguntas clínicas
print("1. Pacientes po grupo étnico y sexo")
df_pacientes_unicos = df_final.drop_duplicates(subset=['PATIENT'])

#RACE y GENDER
tabla_demografica = df_pacientes_unicos.groupby(['RACE', 'GENDER']).size().unstack(fill_value=0)
print(tabla_demografica)

print("\n2. Encuentros con pacientes")
#Encuentros únicos por cada paciente (encounter y patient)
encuentros_por_paciente = df_final.groupby('PATIENT')['ENCOUNTER'].nunique()
media_encuentros = encuentros_por_paciente.mean()
mediana_encuentros = encuentros_por_paciente.median()
print(f"Número medio de encuentros: {media_encuentros:.2f}")
print(f"Número mediano de encuentros: {mediana_encuentros:.2f}")

print("\n3. Top 10 código de observaciones")
#Frecuencia de la combinación código + descripción
top_observaciones = df_final[['CODE', 'DESCRIPTION']].value_counts().head(10).reset_index(name='Frecuencia')
print(top_observaciones.to_string(index=False))

print("\n4. Hemoglobina en sangre (g/dL)")
#Se usa regex=False por los corchetes para que que pandas no lo confunda con una expresión regular
df_hemoglobina = df_final[df_final['DESCRIPTION'].str.contains('Hemoglobin [Mass/volume] in Blood', na=False, regex=False)].copy()

#Se convierte a numérico
df_hemoglobina['VALUE_NUM'] = pd.to_numeric(df_hemoglobina['VALUE'], errors='coerce')

#Distribución
print("Distribución de valores:")
print(df_hemoglobina['VALUE_NUM'].describe())

#Pacientes con al menos 3 mediciones
mediciones_por_paciente = df_hemoglobina.groupby('PATIENT').size()
pacientes_3_o_mas = (mediciones_por_paciente >= 3).sum()
print(f"\nPacientes con al menos 3 mediciones de Hemoglobina: {pacientes_3_o_mas}")

#Cálculo del promedio global vs promedio de promedios
promedio_global = df_hemoglobina['VALUE_NUM'].mean()
promedio_por_paciente = df_hemoglobina.groupby('PATIENT')['VALUE_NUM'].mean()
promedio_de_promedios = promedio_por_paciente.mean()

print(f"\nPromedio global (todas las mediciones): {promedio_global:.2f}")
print(f"Promedio de promedios (un promedio por paciente): {promedio_de_promedios:.2f}")

1. Pacientes po grupo étnico y sexo
GENDER      F    M
RACE              
asian      18   16
black      15   17
hawaiian    3    1
native      0    1
other       4    2
white     218  243

2. Encuentros con pacientes
Número medio de encuentros: 28.08
Número mediano de encuentros: 13.00

3. Top 10 código de observaciones
   CODE                                                                                                   DESCRIPTION  Frecuencia
72514-3                                                 Pain severity - 0-10 verbal numeric rating [Score] - Reported       10490
 8462-4                                                                                      Diastolic Blood Pressure        8229
 8480-6                                                                                       Systolic Blood Pressure        8229
29463-7                                                                                                   Body Weight        7621
 8867-4                     

La media de medias es la metrica correcta para la población, ya que los pacientes enfermos de anemia reportan muchos valores (hay 113 pacientes con más de 3 observaciones) que estarían jalando el promedio, po lo que lo ideal es calcular el promedio por paciente y luego promediar la cohorte, garantizando que cada individuo tenga el mismo peso estadistico.

In [9]:
#Analito objetivo (Hemoglobina)
analito_objetivo = ['Hemoglobin [Mass/volume] in Blood']
df_num = df_final[df_final['DESCRIPTION'].isin(analito_objetivo)].copy()

#Tranformar VALUE a float numérico
df_num['VALUE_NUM'] = pd.to_numeric(df_num['VALUE'], errors='coerce')

#Columna a datetime
df_num['DATE'] = pd.to_datetime(df_num['DATE'], utc=True).dt.tz_localize(None)

#Eliminar la hora
df_num['FECHA_DIA'] = df_num['DATE'].dt.floor('D')

#Reestructurar a formato ancho (Pivot)
df_ancho = df_num.pivot_table(
    index=['PATIENT', 'FECHA_DIA'],
    columns='DESCRIPTION',
    values='VALUE_NUM',
    aggfunc='mean'
).reset_index()

#Valores fuera de rango fisiológico)
rangos_fisiologicos = {
    'Hemoglobin [Mass/volume] in Blood': (5, 25) #Referencia médica: Henry's Clinical Diagnosis and Management by Laboratory Methods (24th ed.).
}

implausibles = {}
for analito, (vmin, vmax) in rangos_fisiologicos.items():
    if analito in df_ancho.columns:
        filas_anomalas = df_ancho[(df_ancho[analito] < vmin) | (df_ancho[analito] > vmax)]
        implausibles[analito] = len(filas_anomalas)

print("Formato ancho (pivot)")
print(df_ancho.head())

print("\nValores implausibles (Manual MSD)")
for analito, conteo in implausibles.items():
    total_mediciones = df_ancho[analito].notnull().sum()
    print(f"{analito}: {conteo} valores implausibles detectados (de {total_mediciones} totales)")

Formato ancho (pivot)
DESCRIPTION                               PATIENT  FECHA_DIA  \
0            00dbff13-8767-26c9-5315-d938617e3e07 2021-11-14   
1            012acae3-d0f3-3493-b3cb-bb430437d0f8 2019-11-30   
2            012acae3-d0f3-3493-b3cb-bb430437d0f8 2025-11-15   
3            0276003c-ce0c-a4cb-b2ed-5443d03cdf46 2019-01-03   
4            0276003c-ce0c-a4cb-b2ed-5443d03cdf46 2025-01-16   

DESCRIPTION  Hemoglobin [Mass/volume] in Blood  
0                                         12.3  
1                                         15.6  
2                                         15.1  
3                                         16.4  
4                                         12.2  

Valores implausibles (Manual MSD)
Hemoglobin [Mass/volume] in Blood: 0 valores implausibles detectados (de 1564 totales)


De acuerdo con McPherson & Pincus, 2021 no hay ningun valor implausible de hemoglobina en sangre en el dataset.

McPherson, R. A., & Pincus, M. R. (2021). Basic Examination of Blood and Bone Marrow. In Henry's Clinical Diagnosis and Management by Laboratory Methods (24th ed., Part 4: Hematology and Transfusion Medicine). Elsevier.

In [10]:
#Actividad 7: Procesar por lotes lo que no cabe
archivo_obs = os.path.join(PATH_CSV, "observations.csv")

#1. Iniciar el perfilador de memoria de Python
tracemalloc.start()

#2. Configurar el tamaño del lote
#10000 filas por iteración, para que la RAM solo se cargue una fraccion del archivo cada ciclo.
TAMANO_LOTE = 10000 

#Diccionarios para acumular la sumatoria y el conteo por cada código
sumas_globales = {}
conteos_globales = {}

#3. Procesamiento por lotes
#Para que no se devuelva un DataFrame completo, usar chunksize, que lo hace por iterador
for chunk in pd.read_csv(archivo_obs, chunksize=TAMANO_LOTE, on_bad_lines='skip'):
    
    #Castear a numérico y descartar nulos
    chunk['VALUE_NUM'] = pd.to_numeric(chunk['VALUE'], errors='coerce')
    chunk_valido = chunk.dropna(subset=['VALUE_NUM'])
    
    #Agrupar localmente en este chunk
    suma_chunk = chunk_valido.groupby('DESCRIPTION')['VALUE_NUM'].sum()
    conteo_chunk = chunk_valido.groupby('DESCRIPTION')['VALUE_NUM'].count()
    
    #Sumar los resultados parciales a nuestros acumuladores globales
    for descripcion, suma in suma_chunk.items():
        sumas_globales[descripcion] = sumas_globales.get(descripcion, 0) + suma
        conteos_globales[descripcion] = conteos_globales.get(descripcion, 0) + conteo_chunk[descripcion]

#4. Calcular la media global a partir de los acumuladores
medias_globales = {
    desc: sumas_globales[desc] / conteos_globales[desc] 
    for desc in sumas_globales if conteos_globales[desc] > 0
}

#5. Capturar el pico máximo de memoria y detener el perfilador
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

pico_mb = peak / (1024 * 1024)
print(f"Memoria RAM limitada (200 MB)")
print(f"Pico máximo de RAM utilizado: {pico_mb:.2f} MB\n")

#Mostrar una muestra de los resultados calculados
df_resultados = pd.DataFrame({
    'Conteo total (N)': conteos_globales,
    'Media global': medias_globales
}).sort_values(by='Conteo total (N)', ascending=False).head(5)

print("Resultados (top 5 analitos)")
print(df_resultados)

Memoria RAM limitada (200 MB)
Pico máximo de RAM utilizado: 2.58 MB

Resultados (top 5 analitos)
                                                    Conteo total (N)  \
Pain severity - 0-10 verbal numeric rating [Sco...             10490   
Systolic Blood Pressure                                         8229   
Diastolic Blood Pressure                                        8229   
Body Weight                                                     7621   
Heart rate                                                      7347   

                                                    Media global  
Pain severity - 0-10 verbal numeric rating [Sco...      2.734414  
Systolic Blood Pressure                               114.481650  
Diastolic Blood Pressure                               78.032835  
Body Weight                                            68.932686  
Heart rate                                             80.925099  


Presupuesto establecido de 200 MB de RAM.
Pico de RAM: 2.56 MB.
Se calcularon las medias globales y conteos de toda la población, verificando que los 5 analitos  coinciden con el análisis general.
Con chunksize y la iteraciones la memoria RAM nunca alberga el dataset completo. Así, la RAM leia un fragmento a la vez, se preocesaba y posteriormente se descartaba para dar lugar a la siguiente iteración.

In [11]:
#Actividad 8: Polarys y Pyspark
#Polars

import time
PATH_CSV = "./"
start_time_pl = time.time()

#1. Carga perezosa con tolerancia a líneas irregulares
lf_obs = pl.scan_csv(PATH_CSV + "observations.csv", ignore_errors=True, truncate_ragged_lines=True)
lf_enc = pl.scan_csv(PATH_CSV + "encounters.csv", ignore_errors=True, truncate_ragged_lines=True)
lf_pat = pl.scan_csv(PATH_CSV + "patients.csv", ignore_errors=True, truncate_ragged_lines=True)

#2. Unión de tablas (observations -> encounters -> patients)
lf_joined = (
    lf_obs
    .join(lf_enc, left_on="ENCOUNTER", right_on="Id", how="left", suffix="_enc")
    .join(lf_pat, left_on="PATIENT", right_on="Id", how="left", suffix="_pat")
)

df_final_pl = lf_joined.collect()

#Pacientes por grupo étnico (RACE) y sexo (GENDER)
print("Demografía con polars")
demografia_pl = (
    df_final_pl.unique(subset=["PATIENT"])
    .group_by(["RACE", "GENDER"])
    .len()
    .sort(["RACE", "GENDER"])
)
print(demografia_pl)

#2. Encuentros por paciente
print("\nEncuentros con polars")
encuentros_pl = (
    df_final_pl.group_by("PATIENT")
    .agg(pl.col("ENCOUNTER").n_unique().alias("n_enc"))
)
media_enc_pl = encuentros_pl["n_enc"].mean()
mediana_enc_pl = encuentros_pl["n_enc"].median()
print(f"Media de encuentros: {media_enc_pl:.2f}")
print(f"Mediana de encuentros: {mediana_enc_pl:.2f}")

#3. Top 10 observaciones
print("\nTop 10 observaciones con polars")
top_10_pl = (
    df_final_pl.group_by(["CODE", "DESCRIPTION"])
    .len()
    .sort("len", descending=True)
    .head(10)
)
print(top_10_pl)

#4. Análisis de Hemoglobina
print("\nHemoglobina con polars")
df_hgb_pl = (
    df_final_pl.filter(pl.col("DESCRIPTION") == "Hemoglobin [Mass/volume] in Blood")
    .with_columns(pl.col("VALUE").cast(pl.Float64, strict=False).alias("VALUE_NUM"))
    .drop_nulls(subset=["VALUE_NUM"])
)

media_global_pl = df_hgb_pl["VALUE_NUM"].mean()
hgb_por_paciente_pl = (
    df_hgb_pl.group_by("PATIENT")
    .agg([
        pl.col("VALUE_NUM").mean().alias("media_paciente"),
        pl.len().alias("conteo")
    ])
)
pacientes_3_mas_pl = hgb_por_paciente_pl.filter(pl.col("conteo") >= 3).height
media_de_medias_pl = hgb_por_paciente_pl["media_paciente"].mean()

print(f"Pacientes con >= 3 mediciones: {pacientes_3_mas_pl}")
print(f"Media global: {media_global_pl:.2f}")
print(f"Media de medias: {media_de_medias_pl:.2f}")

end_time_pl = time.time() - start_time_pl
print(f"\n[Polars - Tiempo de ejecución: {end_time_pl:.2f} s]")

Demografía con polars
shape: (12, 3)
┌────────┬────────┬─────┐
│ RACE   ┆ GENDER ┆ len │
│ ---    ┆ ---    ┆ --- │
│ str    ┆ str    ┆ u32 │
╞════════╪════════╪═════╡
│ null   ┆ null   ┆ 1   │
│ asian  ┆ F      ┆ 18  │
│ asian  ┆ M      ┆ 16  │
│ black  ┆ F      ┆ 15  │
│ black  ┆ M      ┆ 17  │
│ …      ┆ …      ┆ …   │
│ native ┆ M      ┆ 1   │
│ other  ┆ F      ┆ 4   │
│ other  ┆ M      ┆ 2   │
│ white  ┆ F      ┆ 218 │
│ white  ┆ M      ┆ 243 │
└────────┴────────┴─────┘

Encuentros con polars
Media de encuentros: 28.99
Mediana de encuentros: 14.00

Top 10 observaciones con polars
shape: (10, 3)
┌─────────┬─────────────────────────────────┬───────┐
│ CODE    ┆ DESCRIPTION                     ┆ len   │
│ ---     ┆ ---                             ┆ ---   │
│ str     ┆ str                             ┆ u32   │
╞═════════╪═════════════════════════════════╪═══════╡
│ 72514-3 ┆ Pain severity - 0-10 verbal nu… ┆ 10490 │
│ 8462-4  ┆ Diastolic Blood Pressure        ┆ 8229  │
│ 8480-6  ┆ Syst

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, mean, count, expr, desc

start_time_sp = time.time()

#1. Iniciar sesión de Spark local
spark = SparkSession.builder \
    .appName("Biomedical_Pipeline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

#2. Carga de datos
df_obs = spark.read.csv(PATH_CSV + "observations.csv", header=True, inferSchema=True)
df_enc = spark.read.csv(PATH_CSV + "encounters.csv", header=True, inferSchema=True)
df_pat = spark.read.csv(PATH_CSV + "patients.csv", header=True, inferSchema=True)

#Renombrar las columnas conflictivas para hacer el join de forma correcta
df_enc = df_enc.withColumnRenamed("Id", "ENC_Id") \
               .withColumnRenamed("PATIENT", "ENC_PATIENT") \
               .withColumnRenamed("CODE", "ENC_CODE") \
               .withColumnRenamed("DESCRIPTION", "ENC_DESCRIPTION")
df_pat = df_pat.withColumnRenamed("Id", "PAT_Id")

#3. Uniones
df_joined = df_obs.join(df_enc, df_obs.ENCOUNTER == df_enc.ENC_Id, "left") \
                  .join(df_pat, df_obs.PATIENT == df_pat.PAT_Id, "left")

#Pacientes por grupo étnico (RACE) y sexo (GENDER)
print("Demografía con PySpark")
demografia_sp = df_joined.dropDuplicates(["PATIENT"]).groupBy("RACE", "GENDER").count().orderBy("RACE", "GENDER")
demografia_sp.show()

#2. Encuentros por paciente
print("Encuentros con PySpark")
encuentros_sp = df_joined.groupBy("PATIENT").agg(countDistinct("ENCOUNTER").alias("n_enc"))
stats_enc = encuentros_sp.select(
    mean("n_enc").alias("media"),
    expr("percentile_approx(n_enc, 0.5)").alias("mediana")
).collect()[0]
print(f"Media de encuentros: {stats_enc['media']:.2f}")
print(f"Mediana de encuentros: {stats_enc['mediana']:.2f}")

#3. Top 10 Observaciones
print("\nTop 10 observaciones con PySpark")
top_10_sp = df_joined.groupBy("CODE", "DESCRIPTION").count().orderBy(desc("count")).limit(10)
top_10_sp.show(truncate=False)

#4. Análisis de Hemoglobina
print("Hemoglobina con PySpark")
df_hgb_sp = df_joined.filter(col("DESCRIPTION") == "Hemoglobin [Mass/volume] in Blood") \
                     .withColumn("VALUE_NUM", col("VALUE").cast("double")) \
                     .filter(col("VALUE_NUM").isNotNull())

media_global_sp = df_hgb_sp.select(mean("VALUE_NUM")).collect()[0][0]

hgb_por_paciente_sp = df_hgb_sp.groupBy("PATIENT").agg(
    mean("VALUE_NUM").alias("media_paciente"),
    count("VALUE_NUM").alias("conteo")
)

pacientes_3_mas_sp = hgb_por_paciente_sp.filter(col("conteo") >= 3).count()
media_de_medias_sp = hgb_por_paciente_sp.select(mean("media_paciente")).collect()[0][0]

print(f"Pacientes con >= 3 mediciones: {pacientes_3_mas_sp}")
print(f"Media global: {media_global_sp:.2f}")
print(f"Media de medias: {media_de_medias_sp:.2f}")

spark.stop()
end_time_sp = time.time() - start_time_sp
print(f"\n[PySpark - Tiempo de ejecución: {end_time_sp:.2f} s]")

Demografía con PySpark
+--------+------+-----+
|    RACE|GENDER|count|
+--------+------+-----+
|    NULL|  NULL|    1|
|   asian|     F|   18|
|   asian|     M|   16|
|   black|     F|   15|
|   black|     M|   17|
|hawaiian|     F|    3|
|hawaiian|     M|    1|
|  native|     M|    1|
|   other|     F|    4|
|   other|     M|    2|
|   white|     F|  218|
|   white|     M|  243|
+--------+------+-----+

Encuentros con PySpark
Media de encuentros: 28.08
Mediana de encuentros: 13.00

Top 10 observaciones con PySpark
+-------+-------------------------------------------------------------------------------------------------------------+-----+
|CODE   |DESCRIPTION                                                                                                  |count|
+-------+-------------------------------------------------------------------------------------------------------------+-----+
|72514-3|Pain severity - 0-10 verbal numeric rating [Score] - Reported                               

Al comparar los reultados de pandas, polars y pysparks, se obtienen los mismos calculos. Sin embargo, hay una diferencia en el conteo de encuentros:
- Pandas y PySpark reportan una media de 28.08 g/dL, con una mediana de 13.
- Polars reporta una media de 28.99 g/dL, con una mediana 14.
Esta diferencia se debe al metodo de agregación. En Pandas y PySpark se descartan automaticamente los nulos al contar valores únicos, mientras que en polars se incluye a null como un valor único valido, por lo que suma 1 a la cuenta de pacientes.
Con este ejercicio se muestra que no basta con migrar todo nuestro pipeline de pandas a polars o spark solamente traduciendo la sintaxis, es importante saber como el nuevo motor maneja los nulos, ya que varian entre una y otra, en este caso la diferencia no fue grande, pero en otros casos si puede afectar nuestros resultados del analsisi de nuestro dataset, alterando nuestros hallazgos clínicos.

Actividad 9: La recomendación
Considero que para este tipo de análisis, Polars es la mejor opción. Con este dataset que se manejó, los datos caben sin ningun problema en la memoria RAM (en Google Colab tuve problemas a partir de la actividad 6 por el uso de esta). Polars ejecutó el pipeline completo más rápido 0.28 segundos, demostrando que su motor subyacente en Rust y optimizacion de consultas ofrecen el mejor rendimeinto sin emplear demasiada RAM.
Si bien Pandas es perfectamente funcional y el procesameinto con chunk demostró un excelente control de recursos, su naturaleza de carga y bloque de memoria lo hace más lento y acapará más recursos para operaciones como los joins.
PySpark es el menos optimo para este caso, ya que su tiempo y su consumo de recursos son mayores, por lo que Polars o Pandas son mejores opciones.

Por último, la decision entre una otra herramienta depende directamente de los límites del hardware y las caractgeristicas del proyecto. En este caso, trabajamos con un dataset de 20000 pacientes, sin embargo en la vida real los números son mayores (por ejemplo los datos de uno o más hospitales), superando la RAM de la mayoria de los equipos de computo disponibles (y la capacidad de cada uno de manejar más de cierta cantidad de RAM), en ese caso y como hemos visto en calse, los API o el procesamiento por lotes con chunck son buenas opciones, ya que permiten procesar volúmenes más grandes que la memoria disponible sin tener que abandonar el entorno local. La decisión por PySpark sería cuando se trabaja con un volumen grande de datos (por ejemplo los datos clínicos de todo un país), ya que sería imposible almacenar y analizar esos datos de manera local. 